In [ ]:
# Parámetros
plant_codes  = ""
period_start  = None     # Si vacío y incremental=True, se calcula por planta como max(Date) en scada_proc - overlap
period_end    = None

# Modo incremental: para cada planta, reprocesa solo desde max(Date) en scada_proc en adelante.
# Si scada_proc aún no tiene datos para la planta, hace proceso completo la primera vez.
# Pon a False para forzar reproceso completo (period_start sigue siendo override).
incremental   = True
# Días de "overlap": cuántos días previos al max(Date) de scada_proc volver a procesar.
# Mínimo 1 recomendado porque c2 agrupa por día completo — si el último día estaba a medias
# en scada_proc, necesita re-procesarse cuando llegue el resto de horas del día.
overlap_days  = 1

In [ ]:
def _norm_plant_codes(v):
    """Normaliza plant_codes a lista (una o varias). None/'' = todas.
    Acepta lista/tupla, JSON array o cadena separada por comas."""
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        import json as _json
        parsed = _json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, TypeError):
        pass
    return [p.strip() for p in s.split(',') if p.strip()]


import io
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_timestamp

LAKEHOUSE_FILES = '/lakehouse/default/Files'
ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read_sheet(sheet, table=None, header=0):
    """Lee una TABLA ESTRUCTURADA del Excel de config: 1) por NOMBRE de tabla
    (case-insensitive, en cualquier hoja); 2) si no, la (primera) tabla de la hoja
    indicada; 3) ultimo recurso, la hoja completa. Robusto ante renombrados."""
    local = '/tmp/plants_config_proc.xlsx'
    if not Path(local).exists():
        abfs_path = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    import openpyxl
    wb = openpyxl.load_workbook(local, data_only=True)
    def _df(ws, ref):
        _d = [[c.value for c in _r] for _r in ws[ref]]
        return pd.DataFrame(_d[1:], columns=_d[0]) if _d else pd.DataFrame()
    if table:
        for ws in wb.worksheets:
            _tm = {k.lower(): k for k in ws.tables}
            if table.lower() in _tm:
                return _df(ws, ws.tables[_tm[table.lower()]].ref)
    if sheet in wb.sheetnames:
        ws = wb[sheet]
        if ws.tables:
            return _df(ws, ws.tables[next(iter(ws.tables))].ref)
    return pd.read_excel(local, sheet_name=sheet, header=header)
def get_active_plants():
    df = _read_sheet('01_Plants', 'T01_plants')
    df = df[df['pr_execution'].astype(str).str.upper().isin(['TRUE','VERDADERO','1','SÍ','SI'])]
    _pl = _norm_plant_codes(plant_codes)
    if _pl:
        df = df[df['plant_code'].isin(_pl)]
    return df.to_dict(orient='records')


def _bool(v, default=False):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return default
    s = str(v).strip().upper()
    return s in ('TRUE', 'VERDADERO', '1', 'SÍ', 'SI', 'YES', 'Y')


def _getf(row, key, default):
    """Lee un float opcional del row, con default si falta o esta vacio."""
    if key not in row.index:
        return default
    v = row[key]
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _getf_v(v, default):
    """float de un valor suelto, con default si falta o no es numerico."""
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _monthly_irr_thr(plant_code):
    """07_Irrandiance_Thresholds (tabla T07: plant_code, GHI_m1..POA_m12)."""
    poa, ghi = {}, {}
    try:
        df = _read_sheet('07_Irrandiance_Thresholds', 'T07_irradiance_threshold')
        df.columns = [str(c).strip().upper() for c in df.columns]
        sub = df[df['PLANT_CODE'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for m in range(1, 13):
                if f'POA_M{m}' in df.columns:
                    poa[m] = _getf_v(row[f'POA_M{m}'], 0.0)
                if f'GHI_M{m}' in df.columns:
                    ghi[m] = _getf_v(row[f'GHI_M{m}'], 0.0)
    except Exception:
        pass
    return poa, ghi
def get_outlier_bounds():
    """Lee 06_Limits: limites fisicos POR FAMILIA (POA/GHI/DHI/REF/T_AMB).
    Devuelve {familia: {rango_min, rango_max, abrupt_change, dead_threshold}}."""
    try:
        df = _read_sheet('06_Limits', 'T06_limits')
    except Exception:
        df = _read_sheet('04_Outliers')
    fam_pref = [('T_AMB', 'T_AMB'), ('TAMB', 'T_AMB'), ('POA', 'POA'),
                ('GHI', 'GHI'), ('DHI', 'DHI'), ('REF', 'REF')]
    out = {}
    for _, r in df.iterrows():
        param = str(r.get('parametro', '')).strip().upper()
        fam = next((f for pre, f in fam_pref if param.startswith(pre)), None)
        if fam is None:
            continue
        out[fam] = {
            'rango_min':      _getf(r, 'rango_min', None),
            'rango_max':      _getf(r, 'rango_max', None),
            'abrupt_change':  _getf(r, 'abrupt_change', None),
            'dead_threshold': _getf(r, 'dead_threshold', None),
        }
    return out
def _selected_criteria(plant_code):
    """04_Criteria_Selection (tabla T04: plant_code, c1..c9). c1-c7 numerico (-1 off), c8-c9 bool."""
    sel = {f'c{i}': (0.0 if i <= 7 else True) for i in range(1, 10)}
    try:
        sc = _read_sheet('04_Criteria_Selection', 'T04_criteria_selection')
        sc.columns = [str(c).strip().lower() for c in sc.columns]
        sub = sc[sc['plant_code'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for n in range(1, 10):
                ck = f'c{n}'
                if ck in sc.columns:
                    sel[ck] = _getf_v(row[ck], 0.0) if n <= 7 else _bool(row[ck], True)
    except Exception:
        pass
    return sel
def _sensor_cleaning(plant_code, r):
    """Lee 05_Sensor_Outliers (ancho: <sensor>_iqr_k y <sensor>_max_dev por planta).
    Devuelve (max_dev, iqr_k) por familia. Fallback a columnas antiguas de
    03_Criteria si la hoja o la planta no existen (transicion segura)."""
    fams   = {'POA': 'POA', 'GHI': 'GHI', 'DHI': 'DHI', 'REF': 'REF', 'T_AMB': 'TAMB'}
    legacy = {'POA': 'poa_sensor_max_dev', 'GHI': 'ghi_sensor_max_dev',
              'DHI': 'dhi_sensor_max_dev', 'REF': 'ref_sensor_max_dev',
              'T_AMB': 'temp_sensor_max_dev'}
    srow = None
    try:
        so  = _read_sheet('05_Sensor_Outliers', 'T05_sensor_outliers')
        sub = so[so['plant_code'] == plant_code]
        if not sub.empty:
            srow = sub.iloc[0]
    except Exception:
        srow = None
    max_dev, iqr_k = {}, {}
    for fam, pre in fams.items():
        _leg = _getf(r, legacy[fam], 0.04 if fam == 'POA' else 0.0)
        if srow is not None:
            max_dev[fam] = _getf(srow, f'{pre}_max_dev', _leg)
            iqr_k[fam]   = (_getf(srow, f'{pre}_iqr_k', 0.0) if fam != 'T_AMB' else 0.0)
        else:
            max_dev[fam] = _leg
            iqr_k[fam]   = (_getf(r, 'iqr_k', 0.0) if fam != 'T_AMB' else 0.0)
    return max_dev, iqr_k


def get_criteria(plant_code):
    """Config por planta: 03_Criteria_Config + 06_Limits + 05_Sensor_Outliers
    + 04_Criteria_Selection + 07_Irrandiance_Thresholds."""
    df = _read_sheet('03_Criteria_Config', 'T03_criteria_config')
    rows = df[df['plant_code'] == plant_code]
    if rows.empty:
        raise ValueError(f'{plant_code} not found in 03_Criteria_Config')
    r = rows.iloc[0]

    _sc_max_dev, _sc_iqr_k = _sensor_cleaning(plant_code, r)
    _poa_m, _ghi_m = _monthly_irr_thr(plant_code)

    return {
        # --- Limpieza de sensores (05_Sensor_Outliers, por planta y familia) ---
        'sensor_max_dev':            _sc_max_dev,
        'sensor_iqr_k':              _sc_iqr_k,
        'iqr_daytime_min_wm2':       _getf(r, 'iqr_daytime_min_wm2', 20.0),

        # --- Criterios activos por planta (04_Criteria_Selection) ---
        'selected_criteria':         _selected_criteria(plant_code),

        # --- Umbral de irradiancia MENSUAL para c5/c6 (07_Irrandiance_Thresholds) ---
        'monthly_thr_poa':           _poa_m,
        'monthly_thr_ghi':           _ghi_m,

        # --- Limites fisicos por familia para deteccion de anomalias (06_Limits) ---
        'outlier_bounds':            get_outlier_bounds(),

        # --- Albedo: formula seleccionable por planta (REF/GHI o REF/POA) ---
        'albedo_formula':           (lambda _v: _v if _v in ('REF/GHI', 'REF/POA') else 'REF/GHI')(
                                        str(r.get('albedo_formula', '')).strip().upper().replace(' ', '')),

        # --- Factores de produccion garantizada (03_Criteria_Config) ---
        'degradation_factor':       _getf(r, 'degradation_factor', 1.0),
        'availability_factor':      _getf(r, 'availability_factor', 1.0),
        'guaranteed_epi':           _getf(r, 'guaranteed_epi', 0.98),
        'energy_ref_sensor':        (str(r.get('energy_ref_sensor', 'E_GRID_01')).strip()
                                     or 'E_GRID_01'),

        # --- Toggles operacionales ---
        'aggregate_by_hour':        _bool(r.get('aggregate_by_hour') if 'aggregate_by_hour' in r.index else None, False),
        'is_energy':                _bool(r.get('is_energy') if 'is_energy' in r.index else None, False),
    }

In [ ]:
def _suffix(name):
    return name.split('_')[-1] if '_' in name else ''


# =============================================================================
#  Anomaly detection — flagging por sensor antes de cualquier criterio.
#  Tipos:
#    - out_of_range : valor fuera de [min_bound, max_bound]
#    - missing      : NaN
#    - dead         : derivada < dead_thr mientras |valor| > min_significant
#    - abrupt       : |derivada| > abrupt_thr
#  Para cada sensor <col> creamos:
#    - <col>          : valor numerico (NaN si es anomalia)
#    - <col>_flag     : string 'ok' / 'out_of_range' / 'missing' / 'dead' / 'abrupt'
# =============================================================================

def detect_anomalies_for(df, col, min_bound, max_bound, dead_thr, abrupt_thr, min_significant=5.0):
    """Marca anomalias en una columna. Devuelve (series_limpia, series_flag)."""
    v = pd.to_numeric(df[col], errors='coerce')
    flag = pd.Series('ok', index=df.index, dtype=object)

    flag[v.isna()] = 'missing'
    mask_oor = (~v.isna()) & ((v < min_bound) | (v > max_bound))
    flag[mask_oor] = 'out_of_range'

    diff = v.diff().abs()
    mask_dead = (~v.isna()) & (v.abs() > min_significant) & (diff < dead_thr) & (flag == 'ok')
    flag[mask_dead] = 'dead'
    mask_abrupt = (~v.isna()) & (diff > abrupt_thr) & (flag == 'ok')
    flag[mask_abrupt] = 'abrupt'

    # Solo se anula el valor para 'missing' y 'out_of_range' (sin dato usable).
    # 'dead' y 'abrupt' son fallos de sensor pero el valor se CONSERVA: se marca
    # la flag (control interno) y se sigue calculando con el dato real.
    clean = v.where(~flag.isin(['missing', 'out_of_range']), np.nan)
    return clean, flag


def apply_anomaly_detection(df, crit):
    """Aplica detect_anomalies_for a cada sensor usando los limites POR FAMILIA
    de 06_Limits (crit['outlier_bounds'])."""
    bounds = crit.get('outlier_bounds', {})
    fam_pref = [('GHI_', 'GHI'), ('POA_', 'POA'), ('REF_', 'REF'), ('DHI_', 'DHI')]
    irrad_cols = [c for c in df.columns
                  if any(c.upper().startswith(p) for p, _ in fam_pref)
                  and not c.endswith('_filt') and not c.endswith('_flag')]
    for c in irrad_cols:
        fam = next((f for p, f in fam_pref if c.upper().startswith(p)), None)
        b = bounds.get(fam, {})
        clean, flag = detect_anomalies_for(
            df, c,
            min_bound=b.get('rango_min', -6.0) if b.get('rango_min') is not None else -6.0,
            max_bound=b.get('rango_max', 1500.0) if b.get('rango_max') is not None else 1500.0,
            dead_thr=b.get('dead_threshold', 0.0001) if b.get('dead_threshold') is not None else 0.0001,
            abrupt_thr=b.get('abrupt_change', 800.0) if b.get('abrupt_change') is not None else 800.0,
        )
        df[c] = clean
        df[f'{c}_flag'] = flag

    bt = bounds.get('T_AMB', {})
    tamb_cols = [c for c in df.columns
                 if c.upper().startswith('T_AMB') and not c.endswith('_filt') and not c.endswith('_flag')]
    for c in tamb_cols:
        clean, flag = detect_anomalies_for(
            df, c,
            min_bound=bt.get('rango_min', -30.0) if bt.get('rango_min') is not None else -30.0,
            max_bound=bt.get('rango_max', 50.0) if bt.get('rango_max') is not None else 50.0,
            dead_thr=bt.get('dead_threshold', 0.0001) if bt.get('dead_threshold') is not None else 0.0001,
            abrupt_thr=bt.get('abrupt_change', 4.0) if bt.get('abrupt_change') is not None else 4.0,
            min_significant=1.0,
        )
        df[c] = clean
        df[f'{c}_flag'] = flag

    return df
# =============================================================================
#  Outlier IQR (Tukey) por sensor — sobre valores diurnos de todo el periodo.
#  Marca como outlier (NaN + flag 'outlier_iqr') lo que cae fuera de
#  [Q1 - k*IQR, Q3 + k*IQR]. Aplica a GHI/DHI/REF/POA.
# =============================================================================

def apply_iqr_outliers(df, k_map, daytime_min=20.0):
    # k_map: dict {familia: k} (0 = off). Familias: POA/GHI/DHI/REF.
    fam_of = {'POA_': 'POA', 'GHI_': 'GHI', 'DHI_': 'DHI', 'REF_': 'REF'}
    cols = [c for c in df.columns
            if any(c.upper().startswith(f) for f in fam_of)
            and not c.endswith('_filt') and not c.endswith('_flag')]
    for c in cols:
        _fam = next((fam_of[f] for f in fam_of if c.upper().startswith(f)), None)
        k = (k_map.get(_fam, 0.0) if isinstance(k_map, dict) else (k_map or 0.0))
        if not k or k <= 0:
            continue
        s   = pd.to_numeric(df[c], errors='coerce')
        day = s >= daytime_min                      # solo diurno (evita el sesgo de ceros nocturnos)
        dv  = s[day]
        if dv.notna().sum() < 8:
            continue
        q1, q3 = dv.quantile(0.25), dv.quantile(0.75)
        iqr = q3 - q1
        if iqr <= 0:
            continue
        lo, hi = q1 - k * iqr, q3 + k * iqr
        out = day & ((s < lo) | (s > hi))
        if not bool(out.any()):
            continue
        df.loc[out, c] = np.nan                     # outlier IQR SI se descarta
        fcol = f'{c}_flag'
        if fcol in df.columns:
            df.loc[out & (df[fcol] == 'ok'), fcol] = 'outlier_iqr'
    return df


# =============================================================================
#  Cross-sensor outlier filter PROGRESIVO — generico por familia.
#  En cada pasada, si quedan >2 sensores activos y la mayor desviacion relativa
#  a la media supera pct_threshold, se descarta el sensor que mas se desvia (%)
#  y se recalcula la media. Hasta max_passes pasadas.
#  Con 1-2 sensores o pct<=0 -> passthrough (no se descarta nada).
# =============================================================================

def _cross_sensor_outlier(df, cols, pct_threshold, max_passes=3):
    if not cols:
        return []
    raw    = df[cols].astype(float)
    active = raw.notna()
    if pct_threshold and pct_threshold > 0 and len(cols) >= 3:
        for _ in range(max_passes):
            masked   = raw.where(active)
            avg      = masked.mean(axis=1)
            n_active = active.sum(axis=1)
            with np.errstate(divide='ignore', invalid='ignore'):
                reldev = masked.sub(avg, axis=0).abs().div(avg.abs().replace(0, np.nan), axis=0)
            max_dev = reldev.max(axis=1)
            rows = (n_active > 2) & (max_dev > pct_threshold)
            if not bool(rows.any()):
                break
            worst = reldev.fillna(-1.0).idxmax(axis=1)   # sensor mas desviado por fila
            for c in cols:
                deact = rows & (worst == c)
                if bool(deact.any()):
                    active.loc[deact, c] = False
    for c in cols:
        df[f'{c}_filt'] = raw[c].where(active[c])
    return [f'{c}_filt' for c in cols]


def filter_poa_sensors(df, dev_threshold):
    """POA cross-sensor (Plaza II 4%). Solo descarta con 3+ sensores."""
    cols = sorted([c for c in df.columns
                   if c.upper().startswith('POA_')
                   and not c.endswith('_filt') and not c.endswith('_flag')])
    filt_cols = _cross_sensor_outlier(df, cols, dev_threshold)
    if cols:
        df['POA_avg_filt'] = df[filt_cols].astype(float).mean(axis=1, skipna=True)
    else:
        df['POA_avg_filt'] = np.nan
    return df


def filter_ghi_sensors(df, dev_threshold):
    """GHI cross-sensor (manual 5.2.1). Solo descarta con 3+ sensores."""
    cols = sorted([c for c in df.columns
                   if c.upper().startswith('GHI_')
                   and not c.endswith('_filt') and not c.endswith('_flag')])
    filt_cols = _cross_sensor_outlier(df, cols, dev_threshold)
    if cols:
        df['GHI_avg_filt'] = df[filt_cols].astype(float).mean(axis=1, skipna=True)
    else:
        df['GHI_avg_filt'] = np.nan
    return df


def filter_dhi_sensors(df, dev_threshold):
    """DHI cross-sensor (manual 5.2.2). Solo descarta con 3+ sensores."""
    cols = sorted([c for c in df.columns
                   if c.upper().startswith('DHI_')
                   and not c.endswith('_filt') and not c.endswith('_flag')])
    filt_cols = _cross_sensor_outlier(df, cols, dev_threshold)
    if cols:
        df['DHI_avg_filt'] = df[filt_cols].astype(float).mean(axis=1, skipna=True)
    else:
        df['DHI_avg_filt'] = np.nan
    return df


def filter_tamb_sensors(df, dev_threshold):
    """T_amb cross-sensor (% de desviacion). Solo descarta con 3+ sensores."""
    cols = sorted([c for c in df.columns
                   if c.upper().startswith('T_AMB')
                   and not c.endswith('_filt') and not c.endswith('_flag')])
    filt_cols = _cross_sensor_outlier(df, cols, dev_threshold)
    if cols:
        df['T_AMB_avg_filt'] = df[filt_cols].astype(float).mean(axis=1, skipna=True)
    else:
        df['T_AMB_avg_filt'] = np.nan
    return df


def filter_ref_sensors(df, dev_threshold):
    """REF (celda/piranometro de reflejada) cross-sensor progresivo. Solo descarta con 3+."""
    cols = sorted([c for c in df.columns
                   if c.upper().startswith('REF_')
                   and not c.endswith('_filt') and not c.endswith('_flag')])
    filt_cols = _cross_sensor_outlier(df, cols, dev_threshold)
    if cols:
        df['REF_avg_filt'] = df[filt_cols].astype(float).mean(axis=1, skipna=True)
    else:
        df['REF_avg_filt'] = np.nan
    return df


# =============================================================================
#  Albedo — SIEMPRE horizontal:  ALB = REF / GHI
# =============================================================================

def compute_albedo(df, formula='REF/GHI'):
    """Albedo por sufijo de estacion. 'formula' (config 03_Criteria.albedo_formula)
    elige la irradiancia incidente del denominador:
        'REF/GHI' -> ALB_XX = REF_XX / GHI_XX
        'REF/POA' -> ALB_XX = REF_XX / POA_XX
    Convencion (igual que el Excel Plaza II):
        denominador NaN o REF NaN  -> NaN (vacio)
        denominador == 0 o ratio<0 -> 0
    Se crea un ALB_XX por cada sufijo presente a la vez en REF y en la base.
    """
    base = 'POA' if str(formula).strip().upper().replace(' ', '').endswith('POA') else 'GHI'

    base_cols = sorted([c for c in df.columns
                        if c.upper().startswith(f'{base}_')
                        and not c.endswith('_filt') and not c.endswith('_flag')])
    ref_cols  = sorted([c for c in df.columns
                        if c.upper().startswith('REF_')
                        and not c.endswith('_filt') and not c.endswith('_flag')])

    base_map = {_suffix(c): c for c in base_cols}
    ref_map  = {_suffix(c): c for c in ref_cols}
    pairs = sorted(set(base_map) & set(ref_map))

    alb_cols = []
    for sfx in pairs:
        out = f'ALB_{sfx}'
        den_v = df[base_map[sfx]].astype(float)
        ref_v = df[ref_map[sfx]].astype(float)
        valid_inputs = (~den_v.isna()) & (~ref_v.isna())
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = ref_v / den_v.replace(0, np.nan)   # den==0 -> NaN
        alb = ratio.where(ratio >= 0, 0.0)             # den==0 o ratio<0 -> 0
        df[out] = alb.where(valid_inputs, np.nan)      # entradas no numericas -> NaN
        alb_cols.append(out)

    if alb_cols:
        masked = df[alb_cols].where((df[alb_cols] > 0) & (df[alb_cols] < 1))
        df['ALB_avg_filt'] = masked.mean(axis=1, skipna=True)
    else:
        df['ALB_avg_filt'] = np.nan
    return df


# =============================================================================
#  4 Criterios operativos (estilo Plaza II)
# =============================================================================

def _seasonal_poa_threshold(date_series, crit):
    """Threshold POA segun mes (Apr-Sep / Oct-Mar)."""
    month = date_series.dt.month
    return np.where(
        (month >= 4) & (month <= 9),
        crit['poa_threshold_summer_wm2'],
        crit['poa_threshold_winter_wm2']
    )


def compute_criteria(df, crit):
    """9 criterios (IEC 61724-2). c1-c7 numericos en 04_Criteria_Selection (<0 = OFF); c8-c9 TRUE/FALSE.
    c1-c4: salud avg_filt POA/GHI/DHI/REF. 0 = no-null; >0 = avg_filt >= umbral (W/m2).
    c5/c6: nº horas/dia con POA/GHI >= umbral MENSUAL (07_Irrandiance_Thresholds) >= valor c5/c6.
    c7: setpoint >= umbral (kW) -> operador NO limita la potencia.
    c8: sin Wind Alarm (WA != 1). WS_01 (velocidad viento) NO se usa como criterio.
    c9: PF sano (0.98 < PF < 1.0).
    all_valid = producto de los criterios ACTIVOS en 04_Criteria_Selection."""
    sel = crit.get('selected_criteria', {})

    # Umbral de irradiancia MENSUAL por familia (07_Irrandiance_Thresholds)
    poa_m = crit.get('monthly_thr_poa', {})
    ghi_m = crit.get('monthly_thr_ghi', {})
    month = df['Date'].dt.month
    poa_thr = month.map(lambda m: poa_m.get(int(m), 0.0)).astype(float)
    ghi_thr = month.map(lambda m: ghi_m.get(int(m), 0.0)).astype(float)
    df['poa_above_thr'] = (df['POA_avg_filt'].fillna(0).values >= poa_thr.values).astype(int)
    df['ghi_above_thr'] = (df['GHI_avg_filt'].fillna(0).values >= ghi_thr.values).astype(int)

    diffs = df['Date'].diff().dropna()
    dt_h = float(diffs.mode().iloc[0].total_seconds() / 3600) if not diffs.empty else 0.25

    def _thr_eff(ckey, default=0.0):
        v = sel.get(ckey, 0.0)
        return v if (v is not None and v > 0) else default

    # C1-C4: salud del avg_filt. >0 -> exige avg >= umbral; si no -> exige no-null.
    def _avg_crit(col, thr_val):
        s = df[col]
        if thr_val is not None and thr_val > 0:
            return (s.fillna(-1.0) >= thr_val).astype(int)
        return s.notna().astype(int)
    df['c1'] = _avg_crit('POA_avg_filt', sel.get('c1', 0.0))
    df['c2'] = _avg_crit('GHI_avg_filt', sel.get('c2', 0.0))
    df['c3'] = _avg_crit('DHI_avg_filt', sel.get('c3', 0.0))
    df['c4'] = _avg_crit('REF_avg_filt', sel.get('c4', 0.0))

    df['_day'] = df['Date'].dt.date
    # C5: nº horas/dia con POA por encima del umbral mensual >= valor c5
    h_poa = _thr_eff('c5')
    dh_poa = df.groupby('_day')['poa_above_thr'].transform('sum') * dt_h
    df['daily_hours_above'] = dh_poa
    df['c5'] = (dh_poa >= h_poa).astype(int)
    # C6: nº horas/dia con GHI por encima del umbral mensual >= valor c6
    h_ghi = _thr_eff('c6')
    dh_ghi = df.groupby('_day')['ghi_above_thr'].transform('sum') * dt_h
    df['daily_hours_above_ghi'] = dh_ghi
    df['c6'] = (dh_ghi >= h_ghi).astype(int)

    # C7: setpoint >= umbral (kW). Setpoint alto = sin recorte del operador = valido.
    sp_thr = _thr_eff('c7')
    sp_col = next((c for c in df.columns if c.upper().startswith('SETPOINT')), None)
    sp = df[sp_col].astype(float).fillna(0) if sp_col else pd.Series(0.0, index=df.index)
    df['c7'] = (sp >= sp_thr).astype(int)

    # C8: sin alarma de viento (WA != 1). WS_01 = velocidad de viento (dato, no criterio).
    wa_col = next((c for c in df.columns
                   if (c.upper().startswith('WA_') or c.upper() == 'WA')
                   and not c.endswith('_flag') and not c.endswith('_filt')), None)
    wa = df[wa_col].astype(float).fillna(0) if wa_col else pd.Series(0.0, index=df.index)
    df['c8'] = (wa != 1).astype(int)

    # C9: PF sano (0.98 < PF < 1.0); PF exactamente 1.0 = error de medida.
    pf_col = next((c for c in df.columns
                   if (c.upper() == 'PF_01' or c.upper().startswith('PF_'))
                   and not c.endswith('_flag') and not c.endswith('_filt')), None)
    if pf_col is not None:
        pf = df[pf_col].astype(float)
        df['c9'] = ((pf > 0.98) & (pf < 1.0)).astype(int)
    else:
        df['c9'] = 1

    # Validez global = producto de los criterios ACTIVOS.
    #   c1-c7 numericos: activos salvo valor < 0 (off). c8-c9: bool.
    av = pd.Series(1, index=df.index)
    for ck in ('c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7'):
        v = sel.get(ck, 0.0)
        if v is None or v >= 0:
            av = av * df[ck]
    for ck in ('c8', 'c9'):
        if sel.get(ck, True):
            av = av * df[ck]
    df['all_valid'] = av.astype(int)

    df['hours_aggregated'] = np.where(df['all_valid'] == 1, dt_h, 0.0)

    return df.drop(columns=['_day'])

In [ ]:
from datetime import timedelta

spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')


def get_last_proc_date(plant_code):
    if not spark.catalog.tableExists('scada_proc'):
        return None
    row = spark.sql(f"""
        SELECT MAX(Date) AS mx
        FROM scada_proc
        WHERE plant_code = '{plant_code}'
    """).first()
    return row['mx']


def process_plant(plant_code, crit):
    eff_start = period_start
    if eff_start is None and incremental:
        last_date = get_last_proc_date(plant_code)
        if last_date is not None:
            floor_day = pd.Timestamp(last_date).normalize()
            eff_start = (floor_day - timedelta(days=overlap_days)).isoformat()
            print(f"  incremental: reproceso desde {eff_start}  (último proc {last_date}, overlap {overlap_days}d)")
        else:
            print(f"  primera ejecución de {plant_code}: proceso completo")
    elif eff_start is None:
        print(f"  incremental=False: proceso completo")

    where = [f"plant_code = '{plant_code}'"]
    if eff_start:
        where.append(f"Date >= TIMESTAMP'{eff_start}'")
    if period_end:
        where.append(f"Date <= TIMESTAMP'{period_end}'")
    sql = f"SELECT * FROM scada_raw WHERE {' AND '.join(where)} ORDER BY Date"

    sdf = spark.sql(sql)
    df = sdf.toPandas()
    if df.empty:
        return 0
    df = df.drop(columns=['source_file', 'ingested_at'], errors='ignore')
    df = df.sort_values('Date').reset_index(drop=True)

    # ---- Pipeline ----------------------------------------------------------
    # 1) Anomaly detection per sensor (out_of_range / missing / dead / abrupt)
    df = apply_anomaly_detection(df, crit)

    # 1b) Outliers IQR por sensor (diurno) -> descarta valores fuera de Tukey
    df = apply_iqr_outliers(df, crit['sensor_iqr_k'], crit['iqr_daytime_min_wm2'])

    # 2) Cross-sensor outlier filters progresivos (uno por familia). Umbral = 0 -> off.
    df = filter_poa_sensors(df, crit['sensor_max_dev']['POA'])
    df = filter_ghi_sensors(df, crit['sensor_max_dev']['GHI'])
    df = filter_dhi_sensors(df, crit['sensor_max_dev']['DHI'])
    df = filter_ref_sensors(df, crit['sensor_max_dev']['REF'])
    df = filter_tamb_sensors(df, crit['sensor_max_dev']['T_AMB'])

    # 3) Albedo = REF / GHI (siempre horizontal)
    df = compute_albedo(df, crit['albedo_formula'])

    # 4) Criterios operativos C1-C4
    df = compute_criteria(df, crit)

    df['processed_at'] = datetime.utcnow()
    sdf_out = spark.createDataFrame(df)

    table = 'scada_proc'
    if spark.catalog.tableExists(table):
        target = DeltaTable.forName(spark, table)
        # Mapear solo columnas del source (cada planta tiene sensores distintos).
        _set = {col: f"s.`{col}`" for col in sdf_out.columns}
        (target.alias('t')
            .merge(sdf_out.alias('s'),
                   "t.plant_code = s.plant_code AND t.Date = s.Date")
            .whenMatchedUpdate(set=_set)
            .whenNotMatchedInsert(values=_set)
            .execute())
    else:
        (sdf_out.write
            .format('delta')
            .option('mergeSchema', 'true')
            .partitionBy('plant_code')
            .mode('append')
            .saveAsTable(table))

    return len(df)



def get_all_dest_columns():
    """Universo COMPLETO de columnas destino del mapping (todas las plantas)."""
    m = _read_sheet('02_Sensors_Mapping', 'T02_sensor_mapping')
    dests = []
    for _, r in m.iterrows():
        d = str(r['columna_destino']).strip() if pd.notna(r['columna_destino']) else None
        if d and d not in dests:
            dests.append(d)
    return [d for d in dests if d != 'Date']


def ensure_full_schema(table):
    """Garantiza que la tabla tenga TODAS las columnas de sensores del mapping
    (las ausentes se anaden como DOUBLE null). Asi scada_proc tiene tantas
    columnas de sensores como 02_Sensors_Mapping, sin depender de que planta
    se procese ni del orden."""
    if not spark.catalog.tableExists(table):
        return
    existing = set(spark.table(table).columns)
    for d in get_all_dest_columns():
        if d not in existing:
            spark.sql(f"ALTER TABLE {table} ADD COLUMNS (`{d}` DOUBLE)")
            print(f"  schema: +{d} en {table}")


ensure_full_schema('scada_proc')

results = []
for p in get_active_plants():
    plant = p['plant_code']
    print(f"\n-> {plant}")
    try:
        crit = get_criteria(plant)
        n = process_plant(plant, crit)
        print(f"  {n:,} filas escritas en scada_proc")
        results.append({'plant_code': plant, 'rows': n, 'status': 'ok', 'error': None})
    except Exception as e:
        import traceback as _tb
        err = _tb.format_exc()[:2000]
        print(f"  FAILED: {err}")
        results.append({'plant_code': plant, 'rows': 0, 'status': 'failed', 'error': err})

In [5]:
import json, notebookutils
summary = {
    'plants_processed': len(results),
    'ok':     sum(1 for r in results if r['status'] == 'ok'),
    'failed': sum(1 for r in results if r['status'] == 'failed'),
}
print(json.dumps(summary, indent=2))
try:
    from pathlib import Path as _P
    _P('/lakehouse/default/Files/LOGS').mkdir(parents=True, exist_ok=True)
    _P('/lakehouse/default/Files/LOGS/scada_proc_results.json').write_text(json.dumps(results, indent=2, default=str), encoding='utf-8')
except Exception as _e:
    print('no se pudo volcar results:', _e)
notebookutils.notebook.exit(json.dumps(summary))
